# Verificacion empirica del rate limiting de Binance

Objetivo: confirmar con evidencia real que las hipotesis del modelo de rate limiting
implementado en Panzer coinciden con el comportamiento real de la API de Binance.

**Hipotesis a verificar:**
1. Binance devuelve `X-MBX-USED-WEIGHT-1M` en TODAS las respuestas (incluidas 429)
2. El peso reportado por el servidor coincide con los pesos documentados
3. Las ventanas son fijas por minuto (reset al cambiar de minuto)
4. Los pesos variables (depth, klines) escalan con el parametro `limit`
5. HTTP 429 incluye `Retry-After` y codigo `-1003`
6. El limiter de Panzer previene alcanzar el 429

## Setup

In [1]:
import requests
from time import time, sleep
from datetime import datetime, timezone

BASE = "https://api.binance.com"
FAPI = "https://fapi.binance.com"
DAPI = "https://dapi.binance.com"

def raw_get(base: str, endpoint: str, params=None, timeout=10):
    """GET crudo sin limiter, devuelve (status, headers_dict, body, elapsed)."""
    url = base.rstrip('/') + '/' + endpoint.lstrip('/')
    t0 = time()
    resp = requests.get(url, params=params, timeout=timeout)
    elapsed = time() - t0
    rl_headers = {
        k: v for k, v in resp.headers.items()
        if 'weight' in k.lower() or 'retry' in k.lower() or 'order-count' in k.lower()
    }
    try:
        body = resp.json()
    except ValueError:
        body = resp.text
    return resp.status_code, rl_headers, body, elapsed

print("Setup OK")

Setup OK


## Test 1: Headers de rate limit en respuestas normales

**Hipotesis:** Toda respuesta 200 incluye `X-MBX-USED-WEIGHT-1M` con el peso acumulado.

In [2]:
# Una sola peticion ligera para inspeccionar headers
status, headers, body, elapsed = raw_get(BASE, "/api/v3/time")

print(f"Status: {status}")
print(f"Elapsed: {elapsed:.3f}s")
print(f"Body: {body}")
print(f"\nHeaders de rate limit:")
for k, v in headers.items():
    print(f"  {k}: {v}")

assert 'x-mbx-used-weight-1m' in headers, "FALLO: No se encontro x-mbx-used-weight-1m"
print("\n>> CONFIRMADO: x-mbx-used-weight-1m presente en respuesta 200")

Status: 200
Elapsed: 0.315s
Body: {'serverTime': 1772696223398}

Headers de rate limit:
  x-mbx-used-weight: 1
  x-mbx-used-weight-1m: 1

>> CONFIRMADO: x-mbx-used-weight-1m presente en respuesta 200


## Test 2: Verificar pesos documentados vs reportados por servidor

**Hipotesis:** El incremento en `X-MBX-USED-WEIGHT-1M` entre peticiones coincide
con los pesos documentados en `weights.py`.

In [3]:
from panzer.exchanges.binance.weights import get_weight

# Esperamos al inicio de un nuevo minuto para partir de peso conocido
now = time()
wait = 60 - (now % 60) + 0.5  # medio segundo de margen
if wait < 5:
    wait += 60
print(f"Esperando {wait:.1f}s al inicio de nueva ventana...")
sleep(wait)

# Serie de endpoints con pesos conocidos
test_cases = [
    ("spot", BASE, "/api/v3/ping",    None, "ping"),
    ("spot", BASE, "/api/v3/time",    None, "time"),
    ("spot", BASE, "/api/v3/aggTrades", {"symbol": "BTCUSDT", "limit": 1}, "aggTrades"),
    ("spot", BASE, "/api/v3/klines",  {"symbol": "BTCUSDT", "interval": "1m", "limit": 1}, "klines"),
    ("spot", BASE, "/api/v3/avgPrice", {"symbol": "BTCUSDT"}, "avgPrice"),
]

print(f"{'Endpoint':<16} {'Peso esperado':>13} {'Peso servidor':>13} {'Acumulado':>9} {'Match':>5}")
print("-" * 60)

prev_used = 0
all_match = True

for market, base, endpoint, params, label in test_cases:
    expected = get_weight(market, endpoint, params)
    status, hdrs, _, _ = raw_get(base, endpoint, params)
    used = int(hdrs.get('x-mbx-used-weight-1m', 0))
    actual_delta = used - prev_used
    match = actual_delta == expected
    if not match:
        all_match = False
    print(f"{label:<16} {expected:>13} {actual_delta:>13} {used:>9} {'OK' if match else 'DIFF':>5}")
    prev_used = used

if all_match:
    print("\n>> CONFIRMADO: Todos los pesos coinciden con la documentacion")
else:
    print("\n>> ATENCION: Algunos pesos difieren — revisar weights.py")

Esperando 56.9s al inicio de nueva ventana...
Endpoint         Peso esperado Peso servidor Acumulado Match
------------------------------------------------------------
ping                         1             1         1    OK
time                         1             1         2    OK
aggTrades                    4             4         6    OK
klines                       2             2         8    OK
avgPrice                     2             2        10    OK

>> CONFIRMADO: Todos los pesos coinciden con la documentacion


## Test 3: Pesos variables — depth segun limit

**Hipotesis:** El peso de `/api/v3/depth` escala con el parametro `limit`:
- limit 1-100: peso 5
- limit 101-500: peso 25
- limit 501-1000: peso 50

In [4]:
# Esperar nueva ventana
now = time()
wait = 60 - (now % 60) + 0.5
if wait < 5:
    wait += 60
print(f"Esperando {wait:.1f}s al inicio de nueva ventana...")
sleep(wait)

depth_tests = [
    ({"symbol": "BTCUSDT", "limit": 5},    5,  "limit=5"),
    ({"symbol": "BTCUSDT", "limit": 100},   5,  "limit=100"),
    ({"symbol": "BTCUSDT", "limit": 500},  25,  "limit=500"),
    ({"symbol": "BTCUSDT", "limit": 1000}, 50,  "limit=1000"),
]

print(f"{'Params':<16} {'Peso esperado':>13} {'Delta real':>10} {'Acumulado':>9} {'Match':>5}")
print("-" * 57)

prev_used = 0
for params, expected, label in depth_tests:
    status, hdrs, _, _ = raw_get(BASE, "/api/v3/depth", params)
    used = int(hdrs.get('x-mbx-used-weight-1m', 0))
    delta = used - prev_used
    match = delta == expected
    print(f"{label:<16} {expected:>13} {delta:>10} {used:>9} {'OK' if match else 'DIFF':>5}")
    prev_used = used
    if not match:
        print(f"  >> DIVERGENCIA: esperado {expected}, obtenido {delta}")

Esperando 58.6s al inicio de nueva ventana...
Params           Peso esperado Delta real Acumulado Match
---------------------------------------------------------
limit=5                      5          5         5    OK
limit=100                    5          5        10    OK
limit=500                   25         25        35    OK
limit=1000                  50         50        85    OK


## Test 4: Reset de ventana (fixed window)

**Hipotesis:** Al cruzar el limite de minuto, `X-MBX-USED-WEIGHT-1M` vuelve a un
valor bajo (correspondiente solo al peso de la nueva peticion).

In [5]:
# Hacer unas peticiones para acumular peso
for _ in range(3):
    raw_get(BASE, "/api/v3/time")

_, hdrs_before, _, _ = raw_get(BASE, "/api/v3/time")
used_before = int(hdrs_before.get('x-mbx-used-weight-1m', 0))
ts_before = time()

print(f"Peso acumulado antes del reset: {used_before}")
print(f"Timestamp: {datetime.fromtimestamp(ts_before, tz=timezone.utc).isoformat()}")

# Esperar al siguiente minuto
wait = 60 - (ts_before % 60) + 0.5
print(f"Esperando {wait:.1f}s para cruzar al siguiente minuto...")
sleep(wait)

_, hdrs_after, _, _ = raw_get(BASE, "/api/v3/time")
used_after = int(hdrs_after.get('x-mbx-used-weight-1m', 0))
ts_after = time()

print(f"\nPeso tras cruzar ventana: {used_after}")
print(f"Timestamp: {datetime.fromtimestamp(ts_after, tz=timezone.utc).isoformat()}")

minute_before = int(ts_before) // 60
minute_after = int(ts_after) // 60
print(f"\nBucket antes: {minute_before}, bucket despues: {minute_after}")

if used_after < used_before and minute_after > minute_before:
    print("\n>> CONFIRMADO: La ventana se reinicia al cambiar de minuto (fixed window)")
else:
    print("\n>> RESULTADO INESPERADO — revisar si la ventana es rolling")

Peso acumulado antes del reset: 89
Timestamp: 2026-03-05T07:39:03.180418+00:00
Esperando 57.3s para cruzar al siguiente minuto...

Peso tras cruzar ventana: 1
Timestamp: 2026-03-05T07:40:00.755807+00:00

Bucket antes: 29544939, bucket despues: 29544940

>> CONFIRMADO: La ventana se reinicia al cambiar de minuto (fixed window)


## Test 5: Comparacion headers Spot vs Futures UM vs Futures CM

**Hipotesis:** Los tres mercados usan el mismo esquema de headers pero con
limites diferentes (spot=6000/min, futures=2400/min).

In [6]:
markets = [
    ("Spot",       BASE, "/api/v3/time"),
    ("Futures UM", FAPI, "/fapi/v1/time"),
    ("Futures CM", DAPI, "/dapi/v1/time"),
]

for label, base, endpoint in markets:
    status, hdrs, body, elapsed = raw_get(base, endpoint)
    print(f"\n=== {label} ===")
    print(f"  Status: {status}  Elapsed: {elapsed:.3f}s")
    print(f"  Body: {body}")
    print(f"  Headers de rate limit:")
    if hdrs:
        for k, v in hdrs.items():
            print(f"    {k}: {v}")
    else:
        print(f"    (ninguno encontrado)")


=== Spot ===
  Status: 200  Elapsed: 0.258s
  Body: {'serverTime': 1772696400995}
  Headers de rate limit:
    x-mbx-used-weight: 2
    x-mbx-used-weight-1m: 2

=== Futures UM ===
  Status: 200  Elapsed: 0.440s
  Body: {'serverTime': 1772696401397}
  Headers de rate limit:
    x-mbx-used-weight-1m: 1

=== Futures CM ===
  Status: 200  Elapsed: 0.703s
  Body: {'serverTime': 1772696402138}
  Headers de rate limit:
    x-mbx-used-weight-1m: 1


## Test 6: exchangeInfo — rateLimits reales

**Hipotesis:** `/exchangeInfo` devuelve un array `rateLimits` con los tipos
REQUEST_WEIGHT, ORDERS y RAW_REQUESTS.

In [7]:
import json

exchange_info_endpoints = [
    ("Spot",       BASE, "/api/v3/exchangeInfo"),
    ("Futures UM", FAPI, "/fapi/v1/exchangeInfo"),
    ("Futures CM", DAPI, "/dapi/v1/exchangeInfo"),
]

for label, base, endpoint in exchange_info_endpoints:
    status, hdrs, body, _ = raw_get(base, endpoint)
    rate_limits = body.get("rateLimits", []) if isinstance(body, dict) else []
    print(f"\n=== {label} === (status={status})")
    print(f"rateLimits ({len(rate_limits)} entries):")
    for rl in rate_limits:
        print(f"  {json.dumps(rl)}")
    print(f"Headers: {hdrs}")


=== Spot === (status=200)
rateLimits (4 entries):
  {"rateLimitType": "REQUEST_WEIGHT", "interval": "MINUTE", "intervalNum": 1, "limit": 6000}
  {"rateLimitType": "ORDERS", "interval": "SECOND", "intervalNum": 10, "limit": 100}
  {"rateLimitType": "ORDERS", "interval": "DAY", "intervalNum": 1, "limit": 200000}
  {"rateLimitType": "RAW_REQUESTS", "interval": "MINUTE", "intervalNum": 5, "limit": 61000}
Headers: {'x-mbx-used-weight': '22', 'x-mbx-used-weight-1m': '22'}

=== Futures UM === (status=200)
rateLimits (3 entries):
  {"rateLimitType": "REQUEST_WEIGHT", "interval": "MINUTE", "intervalNum": 1, "limit": 2400}
  {"rateLimitType": "ORDERS", "interval": "MINUTE", "intervalNum": 1, "limit": 1200}
  {"rateLimitType": "ORDERS", "interval": "SECOND", "intervalNum": 10, "limit": 300}
Headers: {'x-mbx-used-weight-1m': '2'}

=== Futures CM === (status=200)
rateLimits (2 entries):
  {"rateLimitType": "REQUEST_WEIGHT", "interval": "MINUTE", "intervalNum": 1, "limit": 2400}
  {"rateLimitType":

## Test 7: Limiter de Panzer — previene el 429

**Hipotesis:** Con `safety_ratio=0.9` y `max_per_minute` bajo, el limiter
duerme antes de alcanzar el limite efectivo, evitando el 429.

In [8]:
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL

# Limiter con techo bajo para provocar el sleep rapido
limiter = BinanceFixedWindowLimiter(
    max_per_minute=10,  # muy bajo: effective_limit = 9
    safety_ratio=0.9,
)

print(f"max_per_minute={limiter.max_per_minute}")
print(f"effective_limit={int(limiter.max_per_minute * limiter.safety_ratio)}")
print(f"{'i':>3} {'status':>6} {'elapsed':>8} {'used_local':>10} {'server_used':>11} {'nota'}")
print("-" * 55)

for i in range(12):
    t0 = time()
    try:
        data, headers = binance_public_get(
            base_url=BINANCE_SPOT_BASE_URL,
            endpoint="/api/v3/time",
            params=None,
            limiter=limiter,
            weight=1,
            timeout=5,
        )
        status = 200
        nota = ""
    except Exception as e:
        status = getattr(e, 'status_code', -1)
        nota = str(e)[:40]
    elapsed = time() - t0

    print(
        f"{i:>3} {status:>6} {elapsed:>7.2f}s"
        f" {limiter.used_local:>10} {limiter.last_server_used!s:>11}"
        f" {'<< SLEEP (limiter previno 429)' if elapsed > 2 else ''}{nota}"
    )

max_per_minute=10
effective_limit=9
  i status  elapsed used_local server_used nota
-------------------------------------------------------


2026-03-05 08:40:05  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=23, weight=1, effective_limit=9). Durmiendo 54.85 segundos.


  0    200    0.70s         23          23 
  1    200   55.12s          1           1 << SLEEP (limiter previno 429)
  2    200    0.26s          2           2 
  3    200    0.26s          3           3 
  4    200    0.26s          4           4 
  5    200    0.26s          5           5 
  6    200    0.26s          6           6 
  7    200    0.26s          7           7 
  8    200    0.26s          8           8 


2026-03-05 08:41:02  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=9, weight=1, effective_limit=9). Durmiendo 57.66 segundos.


  9    200    0.26s          9           9 
 10    200   58.03s          1           1 << SLEEP (limiter previno 429)
 11    200    0.36s          2           2 


## Test 8: Evidencia de un 429 real (datos historicos del repo)

El notebook `exceptions.ipynb` capturo un 429 real el 2025-11-27. Estos son
los datos forenses extraidos de esa ejecucion:

```
Status: 429
x-mbx-used-weight-1m: 6320
retry-after: 24
Body: {"code": -1003, "msg": "Too much request weight used; current limit is 6000
       request weight per 1 MINUTE. Please use WebSocket Streams for live updates
       to avoid polling the API."}
```

**Evidencia confirmada:**
- HTTP 429 se produce al superar 6000 de peso en 1 minuto
- `x-mbx-used-weight-1m` sigue presente en la respuesta 429 (6320 > 6000)
- `retry-after: 24` indica segundos hasta el reset de ventana
- Codigo de error `-1003` con mensaje descriptivo
- El header `x-mbx-used-weight` (sin sufijo) tambien presente con mismo valor

## Test 9: Provocar 429 controlado (OPCIONAL — descomentar para ejecutar)

**ATENCION:** Este test hace peticiones masivas y PROVOCARA un 429.
Solo ejecutar si se necesita evidencia fresca. El ban es de ~24s.

In [9]:
# DESCOMENTAR PARA EJECUTAR — PROVOCARA UN 429 REAL
# ===================================================

# from concurrent.futures import ThreadPoolExecutor, as_completed
# import threading
#
# stop = threading.Event()
# evidence = {}
#
# def flood_worker(idx):
#     if stop.is_set():
#         return None
#     status, hdrs, body, elapsed = raw_get(BASE, "/api/v3/exchangeInfo")
#     if status != 200:
#         stop.set()
#         return {"idx": idx, "status": status, "headers": hdrs, "body": body, "elapsed": elapsed}
#     return None
#
# print("Lanzando flood controlado...")
# with ThreadPoolExecutor(max_workers=30) as pool:
#     futs = [pool.submit(flood_worker, i) for i in range(2000)]
#     for f in as_completed(futs):
#         r = f.result()
#         if r:
#             evidence = r
#             break
#
# if evidence:
#     print(f"\n429 CAPTURADO en iteracion {evidence['idx']}")
#     print(f"Status: {evidence['status']}")
#     print(f"Headers: {evidence['headers']}")
#     print(f"Body: {evidence['body']}")
#     print(f"Elapsed: {evidence['elapsed']:.3f}s")
# else:
#     print("No se obtuvo 429 (puede que el limite haya subido)")

print("(Test 9 desactivado — descomentar para ejecutar)")

(Test 9 desactivado — descomentar para ejecutar)


## Resumen de evidencia

In [10]:
print("""
=============================================
RESUMEN DE VERIFICACION EMPIRICA
=============================================

Hipotesis                                      Resultado
----------------------------------------------+---------
1. x-mbx-used-weight-1m en toda respuesta     | Ver Test 1
2. Pesos coinciden con docs                    | Ver Test 2
3. Ventana fija (reset al cambiar minuto)      | Ver Test 4
4. Pesos variables escalan con limit           | Ver Test 3
5. 429 incluye retry-after y code -1003        | Ver Test 8
6. Limiter de Panzer previene 429              | Ver Test 7
7. Spot/UM/CM usan mismo esquema de headers    | Ver Test 5
8. rateLimits de exchangeInfo estructura real   | Ver Test 6

Ejecutar cada test para confirmar resultados.
""")


RESUMEN DE VERIFICACION EMPIRICA

Hipotesis                                      Resultado
----------------------------------------------+---------
1. x-mbx-used-weight-1m en toda respuesta     | Ver Test 1
2. Pesos coinciden con docs                    | Ver Test 2
3. Ventana fija (reset al cambiar minuto)      | Ver Test 4
4. Pesos variables escalan con limit           | Ver Test 3
5. 429 incluye retry-after y code -1003        | Ver Test 8
6. Limiter de Panzer previene 429              | Ver Test 7
7. Spot/UM/CM usan mismo esquema de headers    | Ver Test 5
8. rateLimits de exchangeInfo estructura real   | Ver Test 6

Ejecutar cada test para confirmar resultados.

